# Audio representation (waveform, spectrogram, MFCC) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Waveform to spectrogram to MFCC-like features
A toy sine wave makes the representation ladder visible: raw samples, local spectra, filterbank energies, window effects, and compact cepstral features.

In [ ]:
# 1) waveform samples and energy for a 440 Hz sine
sr=8000; t=np.arange(0,0.01,1/sr); x=np.sin(2*np.pi*440*t); energy=np.mean(x*x)
plt.figure(figsize=(6,3)); plt.plot(t[:40],x[:40],'-o',ms=3); plt.xlabel('seconds'); plt.ylabel('amplitude'); plt.title(f'first samples, energy={energy:.3f}')
assert np.allclose(np.round(x[:5],3),[0.000,0.339,0.637,0.861,0.982]) and abs(energy-0.506)<0.01

In [ ]:
# 2) one Hann-windowed spectrum peaks near 440 Hz
sr=8000; N=256; frame=np.sin(2*np.pi*440*np.arange(N)/sr); mag=np.abs(np.fft.rfft(frame*np.hanning(N))); freqs=np.arange(len(mag))*sr/N; k=int(np.argmax(mag))
plt.figure(figsize=(6,3)); plt.plot(freqs,mag); plt.axvline(freqs[k],c='r',ls='--'); plt.xlim(0,1000); plt.xlabel('Hz'); plt.ylabel('magnitude'); plt.title(f'peak bin {k} = {freqs[k]:.1f} Hz')
assert k==14 and abs(freqs[k]-437.5)<1e-9 and abs(mag[k]-63.489)<0.01

In [ ]:
# 3) MFCC-like filterbank energies and cosine projection
sr=8000; N=256; freqs=np.linspace(0,sr/2,N//2+1); frame=np.sin(2*np.pi*440*np.arange(N)/sr); pow_spec=np.abs(np.fft.rfft(frame*np.hanning(N)))**2
centers=mel_to_hz(np.linspace(hz_to_mel(0),hz_to_mel(sr/2),6)); E=[]
for lo,mid,hi in zip(centers[:-2],centers[1:-1],centers[2:]):
    tri=np.maximum(0,np.minimum((freqs-lo)/(mid-lo),(hi-freqs)/(hi-mid))); E.append(float(pow_spec@tri))
logs=np.log(np.array(E)+1e-9); C=np.array([[math.cos(math.pi*n*(i+.5)/len(logs)) for i in range(len(logs))] for n in range(len(logs))]); ceps=C@logs
plt.figure(figsize=(6,3)); plt.bar(range(4),logs); plt.xlabel('mel-like band'); plt.ylabel('log energy'); plt.title('low bands carry the tone')
assert np.allclose(np.round(E,3),[4631.027,1488.973,0.0,0.0]) and np.allclose(np.round(ceps,3),[-11.397,29.456,-2.149,-8.138])

In [ ]:
# 4) longer windows sharpen frequency resolution
sr=8000; x=np.sin(2*np.pi*440*np.arange(1024)/sr); peaks=[]
plt.figure(figsize=(6,3))
for N in [128,256,512]:
    mag=np.abs(np.fft.rfft(x[:N]*np.hanning(N))); freqs=np.arange(len(mag))*sr/N; peaks.append(freqs[np.argmax(mag)]); plt.plot(freqs,mag/mag.max(),label=f'N={N}')
plt.xlim(300,600); plt.legend(); plt.xlabel('Hz'); plt.title('window length changes bin spacing')
assert abs(peaks[-1]-437.5)<abs(peaks[0]-440)

In [ ]:
# 5) a chirp spectrogram shows frequency moving through time
sr=8000; dur=.25; t=np.arange(0,dur,1/sr); f0=220; f1=880; phase=2*np.pi*(f0*t+(f1-f0)*t*t/(2*dur)); x=np.sin(phase); S=stft_mag(x,256,64); ridge=np.argmax(S,axis=0)*sr/256
plt.figure(figsize=(6,3)); plt.imshow(S,origin='lower',aspect='auto',extent=[0,dur,0,sr/2]); plt.plot(np.linspace(0,dur,len(ridge)),ridge,c='w'); plt.ylim(0,1200); plt.xlabel('seconds'); plt.ylabel('Hz'); plt.title('spectrogram ridge rises')
assert ridge[-1]>ridge[0] and S.shape[0]==129